In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Master/LTAINC/lightweight-medical-model')
DATASET_DIR = PROJECT_DIR / 'data/busi'
OUTPUTS_ROOT = PROJECT_DIR / 'outputs'
RESULTS_ROOT = PROJECT_DIR / 'threshold-sensitivity-results'

assert (PROJECT_DIR / 'tests/test-performance/run_threshold_sensitivity.py').is_file(), 'Thiếu threshold runner'
assert (PROJECT_DIR / 'evaluate_medical_metrics.py').is_file(), 'Thiếu evaluator'
assert DATASET_DIR.is_dir(), f'Không tìm thấy BUSI: {DATASET_DIR}'
print('Project:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Results:', RESULTS_ROOT)

In [ ]:
import os

%cd {PROJECT_DIR}
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
print('PYTHONPATH:', os.environ['PYTHONPATH'])
# Bỏ comment nếu cần cập nhật code trước khi chạy.
# !git pull origin main

# Prediction-refining threshold sensitivity

Notebook này sweep `area_threshold` cho prediction-refining. Mỗi ngưỡng sẽ gọi `evaluate_medical_metrics.py`, sau đó gom `summary.csv` và `per_class_metrics.csv` thành bảng tổng hợp.

In [ ]:
!pip install -q pandas matplotlib scikit-learn pillow

## Cấu hình checkpoint

Đổi `MODEL`, `WIDTH_MULT` và `CHECKPOINT` theo checkpoint bạn muốn sweep. `MODEL` phải là một model có segmentation head: `mk_mnet`, `multitask`, hoặc `r_cbam_mnet`.

In [ ]:
MODEL = 'mk_mnet'
WIDTH_MULT = 0.5

# Ví dụ cho DAMK/MK-MNet nếu bạn đã train và lưu checkpoint ở outputs/busi/mk_mnet/...
CHECKPOINT = OUTPUTS_ROOT / 'busi/mk_mnet/img_224/width_0.5/best_model.pt'

# Ví dụ checkpoint multi-task cũ trong repo này, nếu muốn sweep baseline multitask:
# MODEL = 'multitask'
# WIDTH_MULT = 1.0
# CHECKPOINT = OUTPUTS_ROOT / 'busi/multi/img_224/cbam/seg_weight_1/best_model.pt'

print('Model:', MODEL)
print('Width:', WIDTH_MULT)
print('Checkpoint:', CHECKPOINT)
assert CHECKPOINT.is_file(), f'Không tìm thấy checkpoint: {CHECKPOINT}'

## Chạy sweep

Mặc định các ngưỡng là `1e-4`, `5e-4`, `1e-3`, `5e-3`, `1e-2`, `2e-2`. Dùng `--resume` để bỏ qua các ngưỡng đã có `summary.csv`.

In [ ]:
AREA_THRESHOLDS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 2e-2]
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
MASK_THRESHOLD = 0.5

threshold_args = ' '.join(map(str, AREA_THRESHOLDS))
print('Area thresholds:', threshold_args)

In [ ]:
!python tests/test-performance/run_threshold_sensitivity.py \
  --model "$MODEL" \
  --checkpoint "$CHECKPOINT" \
  --dataset-dir "$DATASET_DIR" \
  --output-root "$RESULTS_ROOT" \
  --evaluator evaluate_medical_metrics.py \
  --image-size $IMAGE_SIZE \
  --width-mult $WIDTH_MULT \
  --batch-size $BATCH_SIZE \
  --num-workers $NUM_WORKERS \
  --mask-threshold $MASK_THRESHOLD \
  --area-thresholds $threshold_args \
  --resume

## Bảng tổng hợp

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(RESULTS_ROOT / 'threshold_sensitivity.csv')
display(summary)

display(summary[[
    'area_threshold', 'accuracy', 'macro_f1', 'macro_auc_ovr',
    'dice_lesion', 'dice_normal', 'normal_specificity',
    'malignant_sensitivity', 'malignant_specificity'
]])

## Hình threshold sensitivity

In [ ]:
from IPython.display import Image, display

plot_path = RESULTS_ROOT / 'threshold_sensitivity.png'
if plot_path.is_file():
    display(Image(filename=str(plot_path)))
else:
    print('Chưa có hình:', plot_path)

## Kiểm tra confusion matrix theo từng ngưỡng

In [ ]:
from IPython.display import Image, display

for matrix_path in sorted(RESULTS_ROOT.glob('area_*/confusion_matrix_refined.png')):
    print(matrix_path.parent.name)
    display(Image(filename=str(matrix_path)))